In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

# =====================================================
# 1️⃣ 데이터 로드 + 시간 파생
# =====================================================
df = pd.read_csv("C:/ai/1month/data/new_flight_weather_merged.csv")

df["departure_datetime"] = pd.to_datetime(df["departure_datetime"])
df["dep_hour"] = df["departure_datetime"].dt.hour
df["dep_weekday"] = df["departure_datetime"].dt.weekday
df["is_weekend"] = df["dep_weekday"].isin([5, 6]).astype(int)

# =====================================================
# ✅ (추가) 도착지 코드화: arrival_code 만들기
# - arrival_code 컬럼이 이미 있으면 정수화
# - 없으면 '도착지'를 숫자화해서 arrival_code 생성
# =====================================================
if "arrival_code" in df.columns:
    df["arrival_code"] = (
        pd.to_numeric(df["arrival_code"], errors="coerce")
          .fillna(-1)
          .astype(int)
    )
elif "도착지" in df.columns:
    df["arrival_code"] = (
        pd.to_numeric(df["도착지"], errors="coerce")
          .fillna(-1)
          .astype(int)
    )
else:
    print("⚠️ '도착지' 또는 'arrival_code' 컬럼이 없습니다. 컬럼명을 확인하세요.")

# =====================================================
# 2️⃣ 상태 → 다중분류 라벨 생성
# =====================================================
label_map = {
    "정상운항": "문제없음",
    "지연": "지연",
    "회항": "회항",
    "취소": "취소"
}

df = df[df["상태"].isin(label_map.keys())].copy()
df["target"] = df["상태"].map(label_map)

print("클래스 분포 (원본)")
print(df["target"].value_counts())

# =====================================================
# 3️⃣ 컬럼 정의 (도착지는 arrival_code로 대체)
# =====================================================
num_cols = ["기온(°C)", "풍속_ms", "dep_hour", "dep_weekday", "is_weekend", ]
num_cols = [c for c in num_cols if c in df.columns]

cat_cols = ["항공사", "출발지", "flight_type","arrival_code"]  # ✅ '도착지' 제거
cat_cols = [c for c in cat_cols if c in df.columns]

for c in cat_cols:
    df[c] = df[c].astype("category")

X_cols = num_cols + cat_cols

print("✅ num_cols:", num_cols)
print("✅ cat_cols:", cat_cols)

# =====================================================
# 4️⃣ Train / Test 분리 (시간 기준)
# =====================================================
df = df.sort_values("departure_datetime")
split_date = df["departure_datetime"].quantile(0.8)

train_df = df[df["departure_datetime"] <= split_date]
test_df  = df[df["departure_datetime"] > split_date]

print("Train:", len(train_df), "Test:", len(test_df))

# =====================================================
# 5️⃣ 🔥 다운사이징 (Train만)
# =====================================================
train_ok     = train_df[train_df["target"] == "문제없음"]
train_delay  = train_df[train_df["target"] == "지연"]
train_divert = train_df[train_df["target"] == "회항"]
train_cancel = train_df[train_df["target"] == "취소"]

base_n = len(train_delay)

train_ok_down    = train_ok.sample(n=base_n, random_state=42)
train_delay_down = train_delay.sample(n=base_n, random_state=42)

train_down = pd.concat([
    train_ok_down,
    train_delay_down,
    train_divert,
    train_cancel
]).sample(frac=1, random_state=42)

print("\n클래스 분포 (다운사이징 후)")
print(train_down["target"].value_counts())

# =====================================================
# 6️⃣ X / y 분리
# =====================================================
X_train = train_down[X_cols]
y_train = train_down["target"]

X_test  = test_df[X_cols]
y_test  = test_df["target"]

# =====================================================
# 7️⃣ LightGBM 다중분류 모델
# =====================================================
lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=4,

    n_estimators=600,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1
)

lgbm.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols
)

print("\n✅ LightGBM 다중분류 학습 완료")
# =====================================================
# 8️⃣ 평가 (✅ threshold 0.8 적용)
# =====================================================
THR = 0.8

# (1) 기본 예측(그냥 argmax)
y_pred_raw = lgbm.predict(X_test)

# (2) 확률 기반 threshold 예측
proba = lgbm.predict_proba(X_test)              # (n_samples, num_class)
classes = lgbm.classes_                         # 클래스 이름 배열
max_prob = proba.max(axis=1)                    # 각 샘플의 최고 확률
pred_idx = proba.argmax(axis=1)                 # 최고 확률 클래스 인덱스
y_pred_best = classes[pred_idx]                 # 최고 확률 클래스 라벨

# 최고 확률이 THR 미만이면 "불확실"로 처리
y_pred_thr = np.where(max_prob >= THR, y_pred_best, "불확실")

print("\n📊 Multiclass Classification Report (기본 argmax)")
print(classification_report(y_test, y_pred_raw))

print(f"\n📊 Multiclass Classification Report (threshold={THR}, 미만=불확실)")
print("✅ 커버리지(확신 예측 비율):", (max_prob >= THR).mean() * 100, "%")
print(classification_report(y_test, y_pred_thr))


C:\Users\Admin\AppData\Local\Temp\ipykernel_13952\860690558.py:9: DtypeWarning: Columns (31,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/ai/1month/data/new_flight_weather_merged.csv")


클래스 분포 (원본)
문제없음    2354238
지연       480982
회항          852
취소          579
Name: target, dtype: int64
✅ num_cols: ['기온(°C)', '풍속_ms', 'dep_hour', 'dep_weekday', 'is_weekend']
✅ cat_cols: ['항공사', '출발지', 'flight_type', 'arrival_code']
Train: 2269321 Test: 567330

클래스 분포 (다운사이징 후)
문제없음    335005
지연      335005
회항         700
취소         464
Name: target, dtype: int64
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013874 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 555
[LightGBM] [Info] Number of data points in the train set: 671174, number of used features: 8
[LightGBM] [Info] Start training from score -0.694883
[LightGBM] [Info] Start training from score -0.694883
[LightGBM] [Info] Start training from score -7.276899
[LightGBM] [Info] Start training from score -6.865703

✅ LightGBM 다중분류 학습 완료

📊 Multiclass Classification Report (기본 ar

C:\Users\Admin\anaconda3\envs\4vector\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Admin\anaconda3\envs\4vector\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        문제없음       0.93      0.20      0.33    421086
         불확실       0.00      0.00      0.00         0
          지연       0.65      0.08      0.14    145977
          취소       0.00      0.00      0.00       115
          회항       0.00      0.00      0.00       152

    accuracy                           0.17    567330
   macro avg       0.32      0.06      0.09    567330
weighted avg       0.86      0.17      0.28    567330



C:\Users\Admin\anaconda3\envs\4vector\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
